In [21]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# ==============================================================================
# 0. CONFIGURATION AND SETUP
# ==============================================================================

# --- Path Setup ---
DATA_PATH = Path('../data')
BASE_DIR = Path('..') 

# --- Define Date Ranges ---
# We want to train on historical data and test strictly on 2024
TRAIN_YEARS = range(2014, 2024)
TEST_YEAR = 2024

# --- Required Columns ---
REQUIRED_COLS_PERFORMANCE = [
    'w_ace', 'l_ace', 'w_df', 'l_df', 'w_svpt', 'l_svpt',
    'w_1stIn', 'l_1stIn', 'w_1stWon', 'l_1stWon',
    'w_2ndWon', 'l_2ndWon', 'w_bpSaved', 'l_bpSaved', 'w_bpFaced', 'l_bpFaced'
]
REQUIRED_COLS_INFO = [
    'winner_name', 'loser_name', 
    'winner_rank', 'loser_rank', 'surface', 'tourney_level', 'tourney_date',
    'winner_age', 'loser_age',
    'winner_ht', 'loser_ht',
    'winner_hand', 'loser_hand'
]
REQUIRED_ALL = REQUIRED_COLS_PERFORMANCE + REQUIRED_COLS_INFO
INITIAL_COLS = ['tourney_id'] + REQUIRED_ALL

print("Setup complete. Ready to load data.")

Setup complete. Ready to load data.


In [22]:
# ==============================================================================
# 0.1 LOAD AND MERGE DATA
# ==============================================================================

matches_list = []

# Gather all files matching the pattern
all_match_files = sorted(list(DATA_PATH.glob('atp_matches_*.csv')))

if not all_match_files:
    raise FileNotFoundError(f"No 'atp_matches_*.csv' files found in {DATA_PATH}.")

print(f"Found {len(all_match_files)} files. Processing...")

for match_file in all_match_files:
    # Filter years: only 2014-2024
    try:
        year_str = match_file.name.split('_')[2].split('.')[0]
        if not year_str.isdigit(): continue
        year = int(year_str)
        if year < 2014 or year > 2024:
            continue
    except:
        continue

    try:
        df = pd.read_csv(match_file, low_memory=False)
        
        # Check for missing columns and fill only performance metrics with NaN if missing
        # (Critical columns like names/ranks must exist)
        missing_cols = [col for col in INITIAL_COLS if col not in df.columns.tolist()]
        
        if not missing_cols:
            matches_list.append(df[INITIAL_COLS])
        else:
            # Handle slight schema variations
            critical_missing = [col for col in INITIAL_COLS if col not in df.columns.tolist() and col not in REQUIRED_COLS_PERFORMANCE]
            if critical_missing:
                print(f"  Skipping {match_file.name}: Missing critical columns {critical_missing}")
            else:
                # Fill non-critical performance cols with NaN
                for col in missing_cols:
                    df[col] = np.nan
                matches_list.append(df[INITIAL_COLS])
                print(f"  Loaded {match_file.name} (filled missing performance cols)")

    except Exception as e:
        print(f"  Error reading {match_file.name}: {e}")

if not matches_list:
    raise FileNotFoundError("No valid matches loaded.")

full_df = pd.concat(matches_list, ignore_index=True)
print(f"Total matches loaded: {len(full_df)}")

Found 11 files. Processing...
Total matches loaded: 30573


In [23]:
# ==============================================================================
# 0.2 CLEANING
# ==============================================================================

initial_rows = len(full_df)

# Drop rows where essential data is missing
full_df.dropna(subset=REQUIRED_ALL, inplace=True) 
dropped_rows = initial_rows - len(full_df)

# Convert date to datetime object
full_df['tourney_date'] = pd.to_datetime(full_df['tourney_date'], format='%Y%m%d', errors='coerce')
full_df.sort_values('tourney_date', inplace=True)

# Identify the Favorite (lower rank is better)
# Note: We handle the rare case of equal ranks by defaulting to winner
full_df['is_winner_favorite'] = full_df['winner_rank'] < full_df['loser_rank']

print(f"Data Cleaned. Rows removed: {dropped_rows}. Final count: {len(full_df)} matches.")

Data Cleaned. Rows removed: 1845. Final count: 28728 matches.


In [24]:
# ==============================================================================
# 1. FEATURE ENGINEERING - PART A: BASIC FEATURES
# ==============================================================================

# Initialize Final DataFrame
final_df = pd.DataFrame()

# --- Target and Context ---
final_df['Target_Win'] = full_df['is_winner_favorite'].map({True: 'Win', False: 'Loss'})
final_df['Surface'] = full_df['surface'].astype(str)
final_df['Tourney_Level'] = full_df['tourney_level'].astype(str)
final_df['Match_Year'] = full_df['tourney_date'].dt.year 

# --- Helper Columns for Logic ---
final_df['Fav_Name'] = full_df.apply(
    lambda row: row['winner_name'] if row['is_winner_favorite'] else row['loser_name'], axis=1
)
final_df['Date'] = full_df['tourney_date']

# --- Rank Difference ---
final_df['Rank_Diff'] = full_df.apply(
    lambda row: row['loser_rank'] - row['winner_rank'] if row['is_winner_favorite'] 
    else row['winner_rank'] - row['loser_rank'], axis=1
)

# --- Physical Stats ---
final_df['Fav_Age'] = full_df.apply(
    lambda row: row['winner_age'] if row['is_winner_favorite'] else row['loser_age'], axis=1
)
final_df['Fav_Height'] = full_df.apply(
    lambda row: row['winner_ht'] if row['is_winner_favorite'] else row['loser_ht'], axis=1
)
final_df['Fav_Hand'] = full_df.apply(
    lambda row: row['winner_hand'] if row['is_winner_favorite'] else row['loser_hand'], axis=1
)

# --- Match Stats Differences (Ace, DF, Service Points) ---
final_df['Ace_Diff'] = full_df.apply(
    lambda row: row['w_ace'] - row['l_ace'] if row['is_winner_favorite'] 
    else row['l_ace'] - row['w_ace'], axis=1
)

final_df['DF_Diff'] = full_df.apply(
    lambda row: row['w_df'] - row['l_df'] if row['is_winner_favorite'] 
    else row['l_df'] - row['w_df'], axis=1
)

# Calculate Service Win % first
full_df['w_service_win_perc'] = (full_df['w_1stWon'] + full_df['w_2ndWon']) / full_df['w_svpt']
full_df['l_service_win_perc'] = (full_df['l_1stWon'] + full_df['l_2ndWon']) / full_df['l_svpt']

final_df['SvPt_Diff'] = full_df.apply(
    lambda row: row['w_service_win_perc'] - row['l_service_win_perc'] if row['is_winner_favorite'] 
    else row['l_service_win_perc'] - row['w_service_win_perc'], axis=1
)
final_df['SvPt_Diff'] = final_df['SvPt_Diff'].replace([np.inf, -np.inf], np.nan)

# --- Hand Matchup Logic ---
def get_hand_matchup(row):
    fav = row['winner_hand'] if row['is_winner_favorite'] else row['loser_hand']
    underdog = row['loser_hand'] if row['is_winner_favorite'] else row['winner_hand']
    
    if fav == underdog: return 'Same'
    if {fav, underdog} <= {'R', 'L'}: return 'Opposite'
    return 'Unknown'

final_df['Hand_Matchup'] = full_df.apply(get_hand_matchup, axis=1)

print("Basic features calculated.")

Basic features calculated.


In [25]:
# ==============================================================================
# 2. FEATURE ENGINEERING - PART B: ADVANCED (Fav_On_Worst_Surface)
# ==============================================================================

# Prepare data for stats: stack winners and losers
winners = full_df[['winner_name', 'surface']].copy()
winners.columns = ['player', 'surface']
winners['result'] = 1 # Win

losers = full_df[['loser_name', 'surface']].copy()
losers.columns = ['player', 'surface']
losers['result'] = 0 # Loss

all_results = pd.concat([winners, losers], axis=0)

# Calculate Win Rate per Player per Surface
surface_stats = all_results.groupby(['player', 'surface'])['result'].agg(['count', 'mean']).reset_index()
surface_stats.rename(columns={'count': 'matches', 'mean': 'win_rate'}, inplace=True)

# Filter: Player needs at least 5 matches on a surface to be considered
surface_stats = surface_stats[surface_stats['matches'] >= 5]

# Find the Worst Surface for each player (min win_rate)
worst_surfaces = surface_stats.loc[surface_stats.groupby('player')['win_rate'].idxmin()]
worst_surface_map = worst_surfaces.set_index('player')['surface'].to_dict()

# Feature Logic
def is_worst_surface(row):
    player = row['Fav_Name']
    current_surface = row['Surface']
    # Retrieve player's worst surface from history
    worst_surf = worst_surface_map.get(player, None) 
    
    if worst_surf is not None and current_surface == worst_surf:
        return 'Yes'
    else:
        return 'No'

final_df['Fav_On_Worst_Surface'] = final_df.apply(is_worst_surface, axis=1)
print("Feature 'Fav_On_Worst_Surface' added.")

Feature 'Fav_On_Worst_Surface' added.


In [26]:
# ==============================================================================
# 3. FEATURE ENGINEERING - PART C: ADVANCED (Fav_Recent_Form)
# ==============================================================================

# 1. Create a chronological list of all matches per player with original index tracking
w_form = full_df[['winner_name', 'tourney_date']].copy()
w_form['original_index'] = full_df.index
w_form.columns = ['player', 'date', 'idx']
w_form['result'] = 1

l_form = full_df[['loser_name', 'tourney_date']].copy()
l_form['original_index'] = full_df.index
l_form.columns = ['player', 'date', 'idx']
l_form['result'] = 0

form_df = pd.concat([w_form, l_form], axis=0)
form_df.sort_values(['player', 'date'], inplace=True)

# 2. Calculate Rolling Win Rate (Last 10 matches)
# shift(1) ensures we use only PAST matches, not the current one
form_df['rolling_win_rate'] = form_df.groupby('player')['result'].transform(
    lambda x: x.shift(1).rolling(window=10, min_periods=1).mean()
)

# 3. Map back to the main DataFrame
# We need to fetch the form specifically for the player who is the FAVORITE in that specific match index
temp_merge = form_df.merge(final_df[['Fav_Name']], left_on='idx', right_index=True)
fav_form_data = temp_merge[temp_merge['player'] == temp_merge['Fav_Name']].set_index('idx')

final_df['raw_form'] = final_df.index.map(fav_form_data['rolling_win_rate'])

# 4. Discretize Form
def discretize_form(win_rate):
    if pd.isna(win_rate):
        return 'Unknown' 
    if win_rate >= 0.80:
        return 'Hot'
    elif win_rate < 0.50:
        return 'Cold'
    else:
        return 'Normal'

final_df['Fav_Recent_Form'] = final_df['raw_form'].apply(discretize_form)

# Final cleanup before splitting
final_df.dropna(subset=['SvPt_Diff', 'raw_form'], inplace=True)
final_df = final_df[final_df['Fav_Recent_Form'] != 'Unknown']

print("Feature 'Fav_Recent_Form' added and discretized.")

Feature 'Fav_Recent_Form' added and discretized.


In [27]:
# ==============================================================================
# 4. SPLITTING AND DISCRETIZATION
# ==============================================================================

# --- Pre-processing: Clean Data before Splitting ---
# Ensure base columns exist and contain valid data (no NaN or Inf)
cols_needed = ['Rank_Diff', 'Ace_Diff', 'DF_Diff', 'SvPt_Diff', 'Fav_Age', 'Fav_Height']
final_df.replace([np.inf, -np.inf], np.nan, inplace=True) 
final_df.dropna(subset=cols_needed, inplace=True) 

# --- Split Train (2014-2023) and Test (2024) ---
df_train_raw = final_df[final_df['Match_Year'] < 2024].copy()
df_test_raw = final_df[final_df['Match_Year'] == 2024].copy()

print(f"Split Complete: Train ({len(df_train_raw)} rows), Test ({len(df_test_raw)} rows)")

# --- 1. Fixed Bins for Rank ---
bins_rank = [-float('inf'), 15, 50, float('inf')]
labels_rank = ['Balanced', 'Moderate', 'Strong']

df_train_raw['Cat_Rank'] = pd.cut(df_train_raw['Rank_Diff'], bins=bins_rank, labels=labels_rank)
df_test_raw['Cat_Rank'] = pd.cut(df_test_raw['Rank_Diff'], bins=bins_rank, labels=labels_rank)

# --- 2. Quantile Bins (Learned on Train, Applied to Test) ---
COLS_TO_BIN = ['Ace_Diff', 'DF_Diff', 'SvPt_Diff', 'Fav_Age', 'Fav_Height']
labels_q = ['Low', 'Medium', 'High']

print("Starting dynamic discretization...")

for col in COLS_TO_BIN:
    # Remove '_Diff' specifically to avoid trailing underscores in new names
    clean_name = col.replace('_Diff', '').replace('Fav_', '').replace('Height', 'Ht')
    new_col = 'Cat_' + clean_name
    
    try:
        # Compute bins on TRAIN data only
        _, bins = pd.qcut(df_train_raw[col], q=3, retbins=True, duplicates='drop')
        
        # Extend bins to infinity
        bins[0] = -float('inf')
        bins[-1] = float('inf')
        
        # Apply bins to both
        df_train_raw[new_col] = pd.cut(df_train_raw[col], bins=bins, labels=labels_q)
        df_test_raw[new_col] = pd.cut(df_test_raw[col], bins=bins, labels=labels_q)
        
        print(f"  [OK] Created column: {new_col}")
        
    except Exception as e:
        print(f"  [ERROR] Could not discretize {col}: {e}")

print("Discretization logic applied.")

Split Complete: Train (25648 rows), Test (2932 rows)
Starting dynamic discretization...
  [OK] Created column: Cat_Ace
  [OK] Created column: Cat_DF
  [OK] Created column: Cat_SvPt
  [OK] Created column: Cat_Age
  [OK] Created column: Cat_Ht
Discretization logic applied.


In [28]:
# ==============================================================================
# 5. FINAL SELECTION AND SAVING
# ==============================================================================

# --- Define Final Nodes ---
NODES = [
    'Surface', 'Tourney_Level', 'Cat_Rank', 'Cat_Age', 'Cat_Ht', 
    'Fav_Hand', 'Hand_Matchup', 'Cat_Ace', 'Cat_DF', 'Cat_SvPt', 
    'Fav_On_Worst_Surface', 'Fav_Recent_Form',
    'Target_Win'
]

# --- Validation ---
# Verify all columns exist in the dataframe before proceeding
missing = [c for c in NODES if c not in df_train_raw.columns]
if missing:
    raise KeyError(f"Missing columns in dataset: {missing}")

# --- Select and Convert ---
df_train_final = df_train_raw[NODES].copy()
df_test_final = df_test_raw[NODES].copy()

# Ensure categorical type for all nodes (crucial for Bayesian Networks)
for col in NODES:
    df_train_final[col] = df_train_final[col].astype('category')
    df_test_final[col] = df_test_final[col].astype('category')

# --- Save Data ---
OUTPUT_PATH = BASE_DIR / 'processed_data'
OUTPUT_PATH.mkdir(exist_ok=True)

TRAIN_FILE = OUTPUT_PATH / 'atp_matches_training.csv'
TEST_FILE = OUTPUT_PATH / 'atp_matches_test_2024.csv'
FULL_FILE = OUTPUT_PATH / 'atp_matches_discretized.csv'

df_train_final.to_csv(TRAIN_FILE, index=False)
df_test_final.to_csv(TEST_FILE, index=False)
# Save combined file just in case, but prefer using split files
pd.concat([df_train_final, df_test_final]).to_csv(FULL_FILE, index=False)

print(f"\n--- PROCESS COMPLETE ---")
print(f"Train Data saved to: {TRAIN_FILE}")
print(f"Test Data saved to:  {TEST_FILE}")
print(f"Columns included:    {NODES}")


--- PROCESS COMPLETE ---
Train Data saved to: ..\processed_data\atp_matches_training.csv
Test Data saved to:  ..\processed_data\atp_matches_test_2024.csv
Columns included:    ['Surface', 'Tourney_Level', 'Cat_Rank', 'Cat_Age', 'Cat_Ht', 'Fav_Hand', 'Hand_Matchup', 'Cat_Ace', 'Cat_DF', 'Cat_SvPt', 'Fav_On_Worst_Surface', 'Fav_Recent_Form', 'Target_Win']
